In [35]:
import os 
import json 
import numpy as np
import pandas as pd
import faiss
from openai import OpenAI
import xgboost as xgb
from dotenv import load_dotenv
import mlflow
from pydantic import BaseModel, Field
from typing import List, Optional
import time

In [2]:
openai_index = faiss.read_index("../rag/openai_faiss.index")
data_sample = pd.read_csv("../rag/products_sample.csv")
hugging_face_index = faiss.read_index("../rag/hf_faiss.index")

print(f"FAISS index loaded: {openai_index.ntotal} products")
print(f"Products loaded: {len(data_sample)}")

FAISS index loaded: 5000 products
Products loaded: 5000


## Lets create retrieve function

In [3]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def retrieve_products(query, k = 5):

    response = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    )
    query_embedding = np.array(response.data[0].embedding)

    # Normalize for cosine similarity

    query_norm = query_embedding / np.linalg.norm(query_embedding)

    #Search FAISS index
    distances, indices = openai_index.search(query_norm.reshape(1, -1).astype("float32"), k)

    results = data_sample.iloc[indices[0]][["parent_asin", "title", "average_rating", "price"]].copy()

    results["similarity_score"] = distances[0]

    return results.reset_index(drop=True)

In [4]:
print(data_sample.columns.tolist())

['parent_asin', 'title', 'description', 'average_rating', 'price', 'text']


In [5]:
test_query = "face cream for dry sensitive skin under $40"
retrieved = retrieve_products(test_query, k=5)
print(retrieved[["title", "average_rating", "price", "similarity_score"]])

                                               title  average_rating  price  \
0  NATURAL Moisturizing Facial Masks (ALOE VERA 4...             4.3    NaN   
1  [DR] DahRuem Relief Ampoule I 1.69 fl.oz I Int...             4.8    NaN   
2  STOCKING STUFFER: BEST UNDER EYE ANTI AGING CR...             3.7    NaN   
3  [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE ...             5.0    NaN   
4  DRENCH Face Moisturizer by Chilogy, Facial Cre...             3.8    NaN   

   similarity_score  
0          0.520070  
1          0.517519  
2          0.516386  
3          0.509013  
4          0.505485  


## lets build the prompt 

In [6]:
def build_prompt(query, retrieved_products, user_history=None):
    
    product_context = ""
    for i, row in retrieved_products.iterrows():
        # Handle NaN price gracefully
        price_str = f"${row['price']:.2f}" if pd.notna(row['price']) else "price not listed"
        rating_str = f"{row['average_rating']:.1f}/5" if pd.notna(row['average_rating']) else "no rating"
        product_context += f"{i+1}. {row['title']} — {rating_str}, {price_str}\n"
    
    history_str = user_history if user_history else "No purchase history available"
    
    prompt = f"""You are a helpful Amazon beauty product advisor.

User is looking for: {query}
User purchase history: {history_str}

Top matching products retrieved:
{product_context}
Based on these products, explain in 3-4 sentences why they match what the user needs.
Be specific — mention product names and ratings.
Highlight the single best pick and explain why.

Return your response as JSON in exactly this format:
{{
    "explanation": "your 3-4 sentence explanation here",
    "top_pick": "exact product name here",
    "reason": "one sentence on why this is the best pick"
}}"""
    
    return prompt

# Test the prompt — print it before sending to LLM
test_retrieved = retrieve_products("face cream for dry sensitive skin", k=5)
print(build_prompt("face cream for dry sensitive skin", test_retrieved))

You are a helpful Amazon beauty product advisor.

User is looking for: face cream for dry sensitive skin
User purchase history: No purchase history available

Top matching products retrieved:
1. [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE - Deep Moisturizing, Soothing Effect for The Sensitive Skin. Natural Superfood Ingredients, Panthenol (Vitamin B5), Beta Glucan, Hyaluronic Acid, 55ml — 5.0/5, price not listed
2. [DR] DahRuem Relief Ampoule I 1.69 fl.oz I Intense Hydration+Moisture ampoule for face with 8% MSM I for Dry and Sensitive Skin — 4.8/5, price not listed
3. NATURAL Moisturizing Facial Masks (ALOE VERA 4PK Box) | Deep Moisturizer Face Sheet Masks for Dry, Sensitive and Tired Skin — 4.3/5, price not listed
4. DRENCH Face Moisturizer by Chilogy, Facial Cream for Oily, Dry, Sensitive Skin with Organic Aloe, Coconut, Neem Oil, Vitamin E for Face, Eyes, Anti Aging Skin Care Products for Men and Women 2 ounces — 3.8/5, price not listed
5. Secura Moisturizing Cream [59431900] 3 o

## let build the LLM pipeline

In [7]:
def generate_explanation(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    raw_text = response.choices[0].message.content
    
    # Strip markdown fences if model adds them
    clean = raw_text.strip().replace("```json", "").replace("```", "").strip()
    
    return json.loads(clean)

# Test it
prompt = build_prompt("face cream for dry sensitive skin", test_retrieved)
result = generate_explanation(prompt)

print("TOP PICK:", result["top_pick"])
print("\nEXPLANATION:", result["explanation"])
print("\nREASON:", result["reason"])

TOP PICK: ANSWER NINETEEN+ THEMOIST SENSITIVE ESSENCE

EXPLANATION: The products listed are specifically formulated for dry and sensitive skin, making them ideal for your needs. For instance, the ANSWER NINETEEN+ THEMOIST SENSITIVE ESSENCE has a perfect rating of 5.0/5 and contains natural superfood ingredients that deeply moisturize and soothe sensitive skin. Additionally, the [DR] DahRuem Relief Ampoule, rated 4.8/5, offers intense hydration with 8% MSM, which is beneficial for dry skin. The Secura Moisturizing Cream, with a solid rating of 4.4/5, is also a great option, especially in a pack of six for extended use.

REASON: This product stands out as the best pick due to its perfect rating of 5.0/5 and its formulation with natural ingredients specifically designed to deeply moisturize and soothe sensitive skin.


## Full pipeline function

In [8]:
def rag_recommend(user_query, user_history=None, k=5):

    # Retrieve
    retrieved = retrieve_products(user_query, k=k)
    
    # Build prompt
    prompt = build_prompt(user_query, retrieved, user_history)
    
    # LLM explanation
    result = generate_explanation(prompt)
    
    # Attach metadata
    result["retrieved_products"] = retrieved[["title", "average_rating", "price", "similarity_score"]].to_dict(orient="records")
    result["query"] = user_query
    
    return result

In [9]:
# Test the full RAG pipeline

result = rag_recommend(
    user_query="face cream for dry sensitive skin under $40",
    user_history="Previously bought: hydrating toner, vitamin C serum"
)

print("QUERY:", result["query"])
print("\nTOP PICK:", result["top_pick"])
print("\nEXPLANATION:", result["explanation"])
print("\nREASON:", result["reason"])
print("\nRETRIEVED PRODUCTS:")
for p in result["retrieved_products"]:
    print(f"  - {p['title'][:60]}... | {p['average_rating']} | {p['similarity_score']:.4f}")

QUERY: face cream for dry sensitive skin under $40

TOP PICK: [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE

EXPLANATION: The products listed cater specifically to dry and sensitive skin, which aligns with your needs. For instance, the [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE has a perfect rating of 5.0/5 and contains natural superfood ingredients like Panthenol and Hyaluronic Acid, making it an excellent choice for deep moisturizing and soothing effects. Additionally, the DRENCH Face Moisturizer by Chilogy, rated 3.8/5, includes organic ingredients like Aloe and Vitamin E, which are beneficial for sensitive skin. Both options are designed to provide hydration without irritation, making them suitable for your skin type.

REASON: This product stands out due to its perfect rating and formulation specifically designed for deep moisturizing and soothing sensitive skin.

RETRIEVED PRODUCTS:
  - NATURAL Moisturizing Facial Masks (ALOE VERA 4PK Box) | Deep... | 4.3 | 0.5200
  - [DR] DahR

## XGBoost Re-Ranking

In [10]:
data = pd.read_csv("../data/amazon_beauty_processed.csv")

print(data.shape)
print(data.columns.tolist())
print(data.head(2))

(75301, 25)
['rating', 'title_x', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'title_y', 'description', 'price', 'average_rating', 'user_idx', 'product_idx', 'user_avg_rating', 'user_review_count', 'user_rating_std', 'product_avg_rating', 'product_review_count', 'product_rating_std', 'rating_deviation', 'review_length', 'is_verified']
   rating                                    title_x  \
0       5  Such a lovely scent but not overpowering.   
1       5  Great for at home use and so easy to use!   

                                                text images        asin  \
0  This spray is really nice. It smells really go...     []  B00YQ6X8EO   
1  This is perfect for my between salon visits. I...     []  B08P2DZB4X   

  parent_asin                       user_id                timestamp  \
0  B00YQ6X8EO  AGKHLEW2SOWHNMFQIJGBECAF7INQ  2020-05-05 14:08:48.923   
1  B08P2DZB4X  AFSKPY37N3C43SOI5IEXEK5JSIYA  2021-07-27 13:04:04.5

In [11]:
# first fetch the feature names from the model to verify they match our data columns

feat = xgb.Booster()
feat.load_model("../ml/xgboost_model.json")
print(feat.feature_names)

['user_avg_rating', 'user_review_count', 'user_rating_std', 'product_avg_rating', 'product_review_count', 'product_rating_std', 'review_length', 'is_verified']


In [12]:
xgb_model = xgb.XGBRegressor()
xgb_model.load_model("../ml/xgboost_model.json")

FEATURE_COLS = [
    "user_avg_rating",
    "user_review_count", 
    "user_rating_std",
    "product_avg_rating",
    "product_review_count",
    "product_rating_std",
    "review_length",
    "is_verified"
]

print("XGBoost model loaded")
print(f"Expected features: {FEATURE_COLS}")

# lets check if the feature names from the model match our expected feature columns as its requires excact same order and names to work properly
if FEATURE_COLS != feat.feature_names:
    print("WARNING: Feature columns do not match model expectations!")
else:
    print("Feature columns match model features.")

XGBoost model loaded
Expected features: ['user_avg_rating', 'user_review_count', 'user_rating_std', 'product_avg_rating', 'product_review_count', 'product_rating_std', 'review_length', 'is_verified']
Feature columns match model features.


In [13]:
def get_user_features(user_id, data):
    
    user_rows = data[data["user_id"] == user_id]
    
    if len(user_rows) == 0:
        return {
            "user_avg_rating": data["user_avg_rating"].mean(),
            "user_review_count": data["user_review_count"].mean(),
            "user_rating_std": data["user_rating_std"].mean(),
            "is_verified": 1
        }
    
    user = user_rows.iloc[0]
    return {
        "user_avg_rating": user["user_avg_rating"],
        "user_review_count": user["user_review_count"],
        "user_rating_std": user["user_rating_std"],
        "is_verified": int(user["is_verified"])
    }

In [14]:
# Building reranking features for the top retrieved products based on user history and product metadata to feed into the XGBoost model for final ranking.

def build_reranking_features(retrieved_products, user_features, data):
    rows = []
    
    for _, product in retrieved_products.iterrows():
        # Try to get product features from processed data first
        product_rows = data[data["parent_asin"] == product["parent_asin"]]
        
        if len(product_rows) > 0:
            # Product has review history — use real features
            prod = product_rows.iloc[0]
            product_avg_rating = prod["product_avg_rating"]
            product_review_count = prod["product_review_count"]
            product_rating_std = prod["product_rating_std"]
        else:
            # No review history — use what we have from metadata directly
            product_avg_rating = product["average_rating"] if pd.notna(product["average_rating"]) else data["product_avg_rating"].mean()
            product_review_count = data["product_review_count"].mean()  # still unknown
            product_rating_std = data["product_rating_std"].mean()      # still unknown
        
        rating_deviation = user_features["user_avg_rating"] - product_avg_rating
        
        rows.append({
            "user_avg_rating": user_features["user_avg_rating"],
            "user_review_count": user_features["user_review_count"],
            "user_rating_std": user_features["user_rating_std"],
            "product_avg_rating": product_avg_rating,
            "product_review_count": product_review_count,
            "product_rating_std": product_rating_std,
            "rating_deviation": rating_deviation,
            "review_length": 200,
            "is_verified": user_features["is_verified"]
        })
    
    return pd.DataFrame(rows, columns=FEATURE_COLS)

In [15]:
# Reranked Pipleline

def rerank_with_xgboost(retrieved_products, user_id, data):
    
    # get user features
    user_features = get_user_features(user_id, data)

    # Build features matrix for the retrieved products
    features_matrix = build_reranking_features(retrieved_products, user_features, data)

    # Score with XGBoost
    ml_scores = xgb_model.predict(features_matrix)

    # Attach scores and rerank
    reranked = retrieved_products.copy().reset_index(drop=True)
    reranked["ml_score"] = ml_scores
    reranked = reranked.sort_values("ml_score", ascending=False).reset_index(drop=True)

    return reranked, user_features

In [19]:
sample_user_id = data["user_id"].iloc[0]

test_retrieved = retrieve_products("face cream for dry sensitive skin", k=5)

print("BEFORE re-ranking (FAISS order):")
print(test_retrieved[["title", "average_rating", "similarity_score"]].to_string())

reranked, user_feats = rerank_with_xgboost(test_retrieved, sample_user_id, data)

print("\nAFTER re-ranking (XGBoost order):")
print(reranked[["title", "average_rating", "similarity_score", "ml_score"]].to_string())

BEFORE re-ranking (FAISS order):
                                                                                                                                                                                                     title  average_rating  similarity_score
0     [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE - Deep Moisturizing, Soothing Effect for The Sensitive Skin. Natural Superfood Ingredients, Panthenol (Vitamin B5), Beta Glucan, Hyaluronic Acid, 55ml             5.0          0.548356
1                                                                          [DR] DahRuem Relief Ampoule I 1.69 fl.oz I Intense Hydration+Moisture ampoule for face with 8% MSM I for Dry and Sensitive Skin             4.8          0.531537
2                                                                              NATURAL Moisturizing Facial Masks (ALOE VERA 4PK Box) | Deep Moisturizer Face Sheet Masks for Dry, Sensitive and Tired Skin             4.3          0.527815
3  DRENCH Face Mois

## Rag Reccomendation

In [22]:
def rag_recommend(user_query, user_id=None, user_history=None, k=5):

    # Step 1: Retrieve from FAISS
    retrieved = retrieve_products(user_query, k=k)
    
    # Step 2: XGBoost re-rank
    if user_id:
        reranked, _ = rerank_with_xgboost(retrieved, user_id, data)
    else:
        reranked = retrieved.copy()
        reranked["ml_score"] = 0.0
    
    # Step 3: Build prompt with re-ranked products
    prompt = build_prompt(user_query, reranked, user_history)
    
    # Step 4: LLM explanation
    result = generate_explanation(prompt)
    
    # Step 5: Attach metadata
    result["retrieved_products"] = reranked[["title", "average_rating", "price", "similarity_score", "ml_score"]].to_dict(orient="records")
    result["query"] = user_query
    result["user_id"] = user_id
    
    return result

# Test end to end
sample_user_id = data["user_id"].iloc[0]

result = rag_recommend(
    user_query="face cream for dry sensitive skin",
    user_id=sample_user_id,
    user_history="Previously bought: hydrating toner, vitamin C serum"
)

print("TOP PICK:", result["top_pick"])
print("\nEXPLANATION:", result["explanation"])
print("\nREASON:", result["reason"])
print("\nRETRIEVED & RERANKED:")
for i, p in enumerate(result["retrieved_products"]):
    print(f"  {i+1}. {p['title'][:55]}... | rating: {p['average_rating']} | ml_score: {p['ml_score']:.4f}")

TOP PICK: [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE

EXPLANATION: The products listed are specifically formulated for dry and sensitive skin, making them ideal for your needs. For instance, the [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE has a perfect rating of 5.0/5 and contains soothing ingredients like Panthenol and Hyaluronic Acid, which are excellent for hydration. Additionally, the [DR] DahRuem Relief Ampoule, rated 4.8/5, offers intense hydration with 8% MSM, further catering to sensitive skin. The Secura Moisturizing Cream, while slightly lower rated at 4.4/5, is also a great option with a bulk pack for ongoing use.

REASON: This product stands out as the best pick due to its perfect rating and inclusion of natural superfood ingredients that deeply moisturize and soothe sensitive skin.

RETRIEVED & RERANKED:
  1. [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE - Deep Mo... | rating: 5.0 | ml_score: 4.9979
  2. [DR] DahRuem Relief Ampoule I 1.69 fl.oz I Intense Hydr... | rat

## Lets do pydantic model to add one more validation layer

In [25]:
class RetrievalProduct(BaseModel):
    title: str
    average_rating: Optional[float] = None
    price: Optional[float] = None
    similarity_score: float
    ml_score: float

class RecommendationResponse(BaseModel):
    query: str
    user_id: Optional[str] = None
    top_pick: str
    explanation: str
    reason: str
    retrieved_products: List[RetrievalProduct]

confidence: Optional[float] = Field(default=0.0, ge=0.0, le=1.0)
category: Optional[str] = None

In [26]:
def generate_explanation(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    raw_text = response.choices[0].message.content
    
    # Strip markdown fences if model adds them
    clean = raw_text.strip().replace("```json", "").replace("```", "").strip()
    
    return json.loads(clean)

    return parsed

def validate_recommendation(result:dict) -> RecommendationResponse:
    try:
        return RecommendationResponse(**result)
    except Exception as e:
        print("Validation error:", e)
        raise ValueError("Generated response does not match expected format.")

In [27]:
# Run full pipeline
result = rag_recommend(
    user_query="face cream for dry sensitive skin",
    user_id=sample_user_id,
    user_history="Previously bought: hydrating toner, vitamin C serum"
)

# Validate
validated = validate_recommendation(result)

# Access fields as typed attributes now — not dict keys
print("TOP PICK:", validated.top_pick)
print("EXPLANATION:", validated.explanation)
print("REASON:", validated.reason)
print("\nPRODUCTS:")
for p in validated.retrieved_products:
    print(f"  - {p.title[:55]}... | {p.average_rating} | {p.ml_score:.4f}")

# Convert back to dict/JSON cleanly
print("\nFULL JSON OUTPUT:")
print(validated.model_dump_json(indent=2))

TOP PICK: [DR] DahRuem Relief Ampoule I 1.69 fl.oz I Intense Hydration+Moisture ampoule for face with 8% MSM I for Dry and Sensitive Skin
EXPLANATION: For your dry and sensitive skin, the [DR] DahRuem Relief Ampoule stands out with its intense hydration and moisture properties, rated 4.8/5. Additionally, the [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE offers deep moisturizing and soothing effects, making it ideal for sensitive skin. The Secura Moisturizing Cream, while a solid option at 4.4/5, comes in a larger pack, which may be beneficial if you're looking for value. However, the ampoule's concentrated formula is particularly effective for immediate relief and hydration.
REASON: This product is the best pick due to its high rating of 4.8/5 and its formulation specifically designed for intense hydration and moisture for dry and sensitive skin.

PRODUCTS:
  - [ANSWER NINETEEN+] THEMOIST SENSITIVE ESSENCE - Deep Mo... | 5.0 | 4.9979
  - [DR] DahRuem Relief Ampoule I 1.69 fl.oz I Inten

## Lets generate synthetic reviews for cold start products

Cold start = products with fewer than 5 reviews in your dataset

In [30]:
product_review_counts = data.groupby("parent_asin").size().reset_index(name="review_count")
cold_start_products = product_review_counts[product_review_counts["review_count"] < 5]

# Get their metadata
cold_start_with_meta = cold_start_products.merge(
    data[["parent_asin", "title_y", "description"]].drop_duplicates("parent_asin"),
    on="parent_asin",
    how="left"
).dropna(subset=["title_y"])

print(f"Total cold start products (< 5 reviews): {len(cold_start_with_meta)}")
print(f"\nSample cold start products:")
print(cold_start_with_meta[["parent_asin", "title_y", "review_count"]].head())

Total cold start products (< 5 reviews): 5433

Sample cold start products:
  parent_asin                                            title_y  review_count
0  1453085815  MAISON MARTIN MARGIELA 'REPLICA' Lazy Sunday M...             3
1  8674394329  BHRINGRAJ POWDER 100% USDA CERTIFIED ORGANIC -...             3
2  979077530X  Versace Man Eau Fraiche 3.4 oz Eau de Toilette...             4
3  B000065AB1  Gillette Mach3 Turbo Cartridges with Aloe & Vi...             4
4  B000068PBL  Norelco 7885XL Quadra Rechargeable Shaving System             4


In [32]:
def generate_synthetic_reviews(product_title, product_description, n=5):

    if isinstance(product_description, list):
        product_description = " ".join(product_description)
    product_description = str(product_description)[:300] if product_description else "No description available"
    
    prompt = f"""Generate {n} realistic Amazon customer reviews for this beauty product.
Make them varied — different ratings (3-5 stars), different writing styles, different use cases.

Product: {product_title}
Description: {product_description}

Return ONLY a JSON array, no other text:
[
    {{"rating": 5, "review": "detailed review text here"}},
    {{"rating": 4, "review": "another review here"}}
]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7  # for creativity and varied reviews
    )
    
    raw = response.choices[0].message.content
    clean = raw.strip().replace("```json", "").replace("```", "").strip()
    reviews = json.loads(clean)
    
    # CRITICAL — always flag synthetic data
    for r in reviews:
        r["is_synthetic"] = True
        r["product_title"] = product_title
    
    return reviews

In [34]:
synthetic_reviews_all = []

sample_cold_start = cold_start_with_meta.head(10)

for _, row in sample_cold_start.iterrows():

    try:
        reviews = generate_synthetic_reviews(product_title = row["title_y"], product_description=row["description"], n=5)

        for r in reviews:

            r["parent_asin"] = row["parent_asin"]
        synthetic_reviews_all.extend(reviews)
        time.sleep(0.5)  # to avoid hitting rate limits

    except Exception as e:
        print(f"Failed for {row['parent_asin']}: {e}")
        continue

sythetic_data = pd.DataFrame(synthetic_reviews_all)
sythetic_data.to_csv("../data/synthetic_reviews.csv", index=False)

print(f"Generated {len(sythetic_data)} synthetic reviews for {len(sample_cold_start)} products.")
print(f"\nSample:")
print(sythetic_data[["product_title", "rating", "is_synthetic", "review"]].head(3).to_string())

Generated 50 synthetic reviews for 10 products.

Sample:
                                                                     product_title  rating  is_synthetic                                                                                                                                                                                                                                                                                review
0  MAISON MARTIN MARGIELA 'REPLICA' Lazy Sunday Morning, Deluze Travel Size.005 oz       5          True  I absolutely love the Lazy Sunday Morning fragrance! It’s light and fresh, perfect for everyday wear. The travel size is super convenient for my handbag, and I find myself reaching for it more than any of my other perfumes. It captures the essence of a cozy morning perfectly!
1  MAISON MARTIN MARGIELA 'REPLICA' Lazy Sunday Morning, Deluze Travel Size.005 oz       4          True             This scent is delightful—very clean and uplifting. I bought 

In [36]:
with mlflow.start_run(run_name="synthetic_review_generation"):
    mlflow.log_param("model", "gpt-4o-mini")
    mlflow.log_param("reviews_per_product", 5)
    mlflow.log_param("cold_start_threshold", 5)
    mlflow.log_param("products_processed", len(sample_cold_start))
    mlflow.log_metric("total_synthetic_reviews", len(sythetic_data))
    mlflow.log_metric("avg_synthetic_rating", sythetic_data["rating"].mean())
    mlflow.log_metric("cold_start_products_total", len(cold_start_with_meta))
    
    # Estimated cost
    estimated_cost = len(sample_cold_start) * 5 * 100 / 1_000_000 * 0.60
    mlflow.log_metric("estimated_cost_usd", round(estimated_cost, 4))
    
    mlflow.log_artifact("../data/synthetic_reviews.csv")

print(f"Logged to MLflow ✓")
print(f"Avg synthetic rating: {sythetic_data['rating'].mean():.2f}")
print(f"Rating distribution:\n{sythetic_data['rating'].value_counts().sort_index()}")

Logged to MLflow ✓
Avg synthetic rating: 4.20
Rating distribution:
rating
3    10
4    20
5    20
Name: count, dtype: int64
